# Comparing Mamba-2 and Gated DeltaNet in MLX

This companion notebook contrasts the selective state-space core of Mamba-2 with the gated DeltaNet blocks used in Qwen 3 Next. Both layers are implemented with MLX primitives so you can poke at their update rules and see how they respond to the same input sequence.

## Architectural intuition

**Mamba-2** wraps a selective state-space model (SSM) around a depthwise convolution. For each token $x_t$ it computes a filtered input $\tilde{x}_t$ and a positive step size $\Delta_t$. The hidden state update resembles a controllable linear dynamical system:

$$h_t = e^{\Delta_t A} \odot h_{t-1} + \Delta_t \cdot (B \tilde{x}_t)$$
$$y_t = C h_t + D \tilde{x}_t, \qquad o_t = \sigma(W_g x_t) \odot y_t$$

The matrices $A, B, C, D$ are learned (often diagonal or low-rank), and the gate $\sigma(W_g x_t)$ selects how much of the SSM response to expose. Because the recurrence is linear in the hidden state, the layer captures long-range dynamics without quadratic attention cost.

**Gated DeltaNet** (Qwen 3 Next) keeps attention-like key/value mixing but replaces the softmax kernel with delta-rule integration. Each layer performs grouped 1-D convolution over concatenated $q, k, v$ streams, normalises the projected queries/keys with zero-centred RMSNorm, and applies a learned decay $A$ and bias $\Delta b$ inside the delta update:

$$(q_t, k_t, v_t, z_t) = W_{qkvz} x_t, \quad (b_t, a_t) = W_{ba} x_t$$
$$\hat{q}_t = \operatorname{RMSNorm}(q_t), \; \hat{k}_t = \operatorname{RMSNorm}(k_t)$$
$$u_t = \hat{q}_t \odot \hat{k}_t, \qquad v_t' = v_t \odot \sigma(a_t)$$
$$y_t, \text{state}_t = \operatorname{DeltaUpdate}(u_t, v_t', b_t, A, \Delta)$$
$$o_t = \operatorname{RMSNorm}(y_t, z_t)$$

Instead of forming attention scores and normalising with softmax, DeltaNet keeps a running weighted sum where the decay $A$ controls how quickly past contributions fade. The final residual mixes in an auxiliary $z_t$ branch.

## Key differences

| Aspect | Mamba-2 selective SSM | Qwen 3 Next Gated DeltaNet |
| --- | --- | --- |
| Core operation | Linear state update $h_t = e^{\Delta_t A} h_{t-1} + \Delta_t B \tilde{x}_t$ | Delta-rule accumulation over grouped $q, k, v$ projections |
| Source of context | Depthwise convolution + recurrent scan | Depthwise convolution + learned decay along value heads |
| Gating | Output gate $\sigma(W_g x_t)$ controls exposure of SSM response | Separate $a_t$ gate modulates values before delta accumulation |
| Normalisation | Optional RMSNorm before projections; emphasis on stability via $A$ parameterisation | Zero-centred RMSNorm on $q$ and $k$ plus gated residual RMSNorm |
| Parallelism | Sequential scan over sequence length (can be parallelised with selective scan kernels) | Fully parallel except for lightweight cumulative delta update |
| Typical usage | Replacement for attention in long-context, low-latency models (e.g. Mamba-2, RWKV variants) | Linear blocks alternating with attention in Qwen 3 Next (3 DeltaNet : 1 attention) |


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Tuple

try:
    from mlx_qwen3_next import (
        Qwen3NextConfig,
        Qwen3NextGatedDeltaNet,
        require_mlx,
    )
except Exception as exc:
    raise RuntimeError("Install MLX and ensure `mlx_qwen3_next.py` is importable before running this notebook.") from exc

require_mlx()

import mlx.core as mx
import mlx.nn as nn


In [ ]:
class ToyMamba2(nn.Module):
    """Minimal selective SSM block inspired by Mamba-2."""

    def __init__(self, hidden_size: int, state_size: int, conv_kernel: int = 4):
        super().__init__()
        self.hidden_size = hidden_size
        self.state_size = state_size
        self.conv_kernel = conv_kernel
        self.in_proj = nn.Linear(hidden_size, hidden_size * 2, bias=False)
        self.delta_proj = nn.Linear(hidden_size, state_size, bias=True)
        self.conv1d = nn.Conv1d(
            in_channels=hidden_size,
            out_channels=hidden_size,
            kernel_size=conv_kernel,
            groups=hidden_size,
            bias=False,
            padding=0,
        )
        self.input_state = nn.Linear(hidden_size, state_size, bias=False)
        self.output_state = nn.Linear(state_size, hidden_size, bias=False)
        self.skip_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.out_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.A_log = mx.random.uniform(low=-2.0, high=0.0, shape=(state_size,))

    def __call__(self, inputs: mx.array) -> mx.array:
        B, S, D = inputs.shape
        features, gate = mx.split(self.in_proj(inputs), 2, axis=-1)
        gate = nn.silu(gate)
        conv_cache = mx.zeros((B, self.conv_kernel - 1, D), dtype=inputs.dtype)
        filtered = self.conv1d(mx.concatenate([conv_cache, features], axis=1))
        delta = nn.softplus(self.delta_proj(inputs))
        state = mx.zeros((B, self.state_size), dtype=inputs.dtype)
        outputs = []
        A = -mx.exp(self.A_log)
        for t in range(S):
            dt = delta[:, t]
            u = filtered[:, t]
            bu = self.input_state(u)
            state = mx.exp(dt * A) * state + dt * bu
            y = self.output_state(state) + self.skip_proj(u)
            outputs.append(y)
        y = mx.stack(outputs, axis=1)
        return self.out_proj(gate * y)


In [ ]:
# Build a small configuration so both blocks can run on CPU-only MLX
config = Qwen3NextConfig.from_dict({
    "hidden_size": 64,
    "linear_num_value_heads": 8,
    "linear_num_key_heads": 4,
    "linear_key_head_dim": 8,
    "linear_value_head_dim": 8,
    "linear_conv_kernel_dim": 4,
    "rms_norm_eps": 1e-6,
})

mamba_block = ToyMamba2(hidden_size=64, state_size=96, conv_kernel=4)
deltanet_block = Qwen3NextGatedDeltaNet(config)

mx.random.seed(7)
inputs = mx.random.normal(shape=(2, 6, 64))

mamba_out = mamba_block(inputs)
deltanet_out = deltanet_block(inputs)

print(f"Mamba-2 block output shape: {mamba_out.shape}")
print(f"Gated DeltaNet output shape: {deltanet_out.shape}")
print(f"Sample Mamba-2 activations:\n{mamba_out[0, :2, :6]}")
print(f"Sample DeltaNet activations:\n{deltanet_out[0, :2, :6]}")


The toy setup processes a batch of two six-token sequences. Both modules map the hidden dimension back to 64, but the activations differ because the Mamba-2 recurrence carries a full state vector while DeltaNet keeps a running decay-weighted mix of the value heads.

## Practical trade-offs

* **Context length & latency:** Mamba-2 excels when you need very long contexts with low latency because the SSM scan avoids quadratic attention cost. DeltaNet relies on grouped convolutions and per-head decays, so it benefits from parallel execution but still mixes information within a local window before the delta update.
* **Compatibility with attention stacks:** Qwen 3 Next alternates DeltaNet with gated attention to keep global token mixing. Mamba-2 is often deployed as a full attention replacement, so it needs carefully tuned positional encodings and residual scaling to remain stable.
* **Implementation complexity:** Selective-scan kernels (as used in the official Mamba-2 release and MLX-LM) require custom fused ops for best performance. DeltaNet can be built from standard convolutions, elementwise gating, and simple recurrence, which maps cleanly to MLX primitives.
* **Quantisation & deployment:** The DeltaNet block shares many traits with attention (linear projections, grouped heads), so existing quantisation tooling applies easily. Mamba-2's state matrices and scan need bespoke quantisation-aware conversions.

## Next steps

* Try increasing the hidden/state sizes or sequence length to see how the activations diverge.
* Swap the toy parameters with those from the [mlx-lm](https://github.com/ml-explore/mlx-examples/tree/main/llms/mlx_lm) Mamba-2 implementation to match production initialisation.
* Integrate the `ToyMamba2` block into the decoder loop in `mlx_qwen3_next.py` to experiment with hybrid Mamba/DeltaNet stacks.